# 01 - Data Ingestion and Schema Review

## Purpose

This notebook loads the source workbook, standardizes sheet and column names, reviews schema quality, checks duplicate borrower IDs, and creates the first merged analytical dataset.

**Professional correction:** `user_id` is not unique in the source workbook. Therefore, this notebook does **not** merge on `user_id` alone. It creates a `record_sequence` within each `user_id` and merges on `[user_id, record_sequence]` to avoid many-to-many row inflation.

In [34]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from credit_risk.data.ingestion import (
    read_source_workbook,
    standardize_workbook_sheets,
    normalize_id_columns,
    merge_credit_risk_sheets,
    merge_credit_risk_sheets_many_to_many_for_audit,
)
from credit_risk.data.schema import summarize_workbook, duplicate_id_summary
from credit_risk.data.validation import validate_expected_columns, validate_target

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Credit_Risk_Dataset.xlsx"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH

WindowsPath('D:/Banking and Finance/Projects/canadian-retail-credit-risk-xai/data/raw/Credit_Risk_Dataset.xlsx')

## Load source workbook

In [35]:
raw_sheets = read_source_workbook(RAW_PATH)

source_shape_summary = pd.DataFrame(
    [{"sheet": sheet_name, "row_count": df.shape[0], "column_count": df.shape[1]} for sheet_name, df in raw_sheets.items()]
)
source_shape_summary

,sheet,row_count,column_count
0,loan_information,134417,5
1,Employment,134417,7
2,Personal_information,134417,8
3,Other_information,134417,7


## Standardize sheet and column names

In [36]:
sheets = standardize_workbook_sheets(raw_sheets)
sheets = normalize_id_columns(sheets)

for sheet_name, df in sheets.items():
    print(f"\n{sheet_name}")
    print(df.columns.tolist())


loan_information
['user_id', 'loan_category', 'amount', 'interest_rate', 'tenure_years']

employment
['user_id', 'employment_type', 'tier_of_employment', 'industry', 'role', 'work_experience', 'total_income_pa']

personal_information
['user_id', 'gender', 'married', 'dependents', 'home', 'pincode', 'social_profile', 'is_verified']

other_information
['user_id', 'delinq_2yrs', 'total_payment', 'received_principal', 'interest_received', 'number_of_loans', 'defaulter']


## Schema summary

In [37]:
schema_summary = summarize_workbook(sheets)
schema_summary.to_csv(TABLE_DIR / "schema_summary.csv", index=False)
schema_summary

,dataset,column,dtype,non_null_count,missing_count,missing_pct,unique_values
0,loan_information,user_id,int64,134417,0,0.00,133752
1,loan_information,loan_category,object,134417,0,0.00,7
2,loan_information,amount,float64,106798,27619,20.55,86157
3,loan_information,interest_rate,float64,134417,0,0.00,137
4,loan_information,tenure_years,int64,134417,0,0.00,2
5,employment,user_id,int64,134417,0,0.00,133752
6,employment,employment_type,object,54731,79686,59.28,2
7,employment,tier_of_employment,object,54731,79686,59.28,7
8,employment,industry,object,134413,4,0.00,12974
9,employment,role,object,134417,0,0.00,46


## Expected-column validation

In [38]:
expected_column_check = validate_expected_columns(sheets)
expected_column_check.to_csv(TABLE_DIR / "expected_column_check.csv", index=False)
expected_column_check

,dataset,expected_column,present
0,loan_information,user_id,True
1,loan_information,loan_category,True
2,loan_information,amount,True
3,loan_information,interest_rate,True
4,loan_information,tenure_years,True
5,employment,user_id,True
6,employment,employment_type,True
7,employment,tier_of_employment,True
8,employment,industry,True
9,employment,role,True


## Duplicate borrower ID review

Duplicate `user_id` values are expected in this workbook. They appear to represent multiple observations/loans for the same borrower rather than simple duplicate rows. This means the analytical grain is **observation-level**, not purely borrower-level.

In [39]:
duplicate_summary = duplicate_id_summary(sheets)
duplicate_summary.to_csv(TABLE_DIR / "duplicate_id_summary.csv", index=False)
duplicate_summary

,dataset,id_column_present,row_count,unique_id_count,duplicate_id_count
0,loan_information,True,134417,133752,665
1,employment,True,134417,133752,665
2,personal_information,True,134417,133752,665
3,other_information,True,134417,133752,665


In [40]:
duplicate_sets = {
    sheet_name: set(df.loc[df["user_id"].duplicated(keep=False), "user_id"])
    for sheet_name, df in sheets.items()
}

same_duplicate_ids_across_sheets = len({frozenset(ids) for ids in duplicate_sets.values()}) == 1

duplicate_alignment_check = pd.DataFrame(
    {
        "check": [
            "same_duplicate_user_ids_across_all_sheets",
            "max_records_per_user_id",
        ],
        "result": [
            same_duplicate_ids_across_sheets,
            max(df["user_id"].value_counts().max() for df in sheets.values()),
        ],
    }
)

duplicate_alignment_check

,check,result
0,same_duplicate_user_ids_across_all_sheets,True
1,max_records_per_user_id,2


## Safe merge source sheets

A many-to-many merge on `user_id` alone would inflate the dataset. The safe merge below preserves source row order within each duplicated `user_id` by adding `record_sequence` and using a one-to-one merge on `[user_id, record_sequence]`.

In [41]:
merged_df = merge_credit_risk_sheets(sheets)

merge_audit = {
    "source_row_count": next(iter(sheets.values())).shape[0],
    "merged_row_count": merged_df.shape[0],
    "merged_column_count": merged_df.shape[1],
    "record_key_duplicate_count": int(merged_df[["user_id", "record_sequence"]].duplicated().sum()),
    "user_id_duplicate_row_count": int(merged_df["user_id"].duplicated().sum()),
}

merge_audit

{'source_row_count': 134417,
 'merged_row_count': 134417,
 'merged_column_count': 25,
 'record_key_duplicate_count': 0,
 'user_id_duplicate_row_count': 665}

In [42]:
# Optional audit only: this shows why we avoid a direct many-to-many merge.
many_to_many_audit_df = merge_credit_risk_sheets_many_to_many_for_audit(sheets)

many_to_many_comparison = pd.DataFrame(
    [
        {
            "merge_method": "safe_sequence_merge",
            "row_count": merged_df.shape[0],
            "default_rate": merged_df["defaulter"].mean(),
        },
        {
            "merge_method": "direct_user_id_many_to_many_audit_only",
            "row_count": many_to_many_audit_df.shape[0],
            "default_rate": many_to_many_audit_df["defaulter"].mean(),
        },
    ]
)

many_to_many_comparison.to_csv(TABLE_DIR / "merge_method_comparison.csv", index=False)
many_to_many_comparison

,merge_method,row_count,default_rate
0,safe_sequence_merge,134417,0.090413
1,direct_user_id_many_to_many_audit_only,143727,0.093712


## Target validation

The target should be binary and non-missing before downstream data-quality, EDA, and modelling notebooks.

In [43]:
target_check = validate_target(merged_df, target="defaulter")
target_check

{'target_present': True,
 'row_count': 134417,
 'target_missing_count': 0,
 'target_values': [0, 1],
 'default_rate': 0.0904126710163149}

## Save interim merged dataset

In [44]:
merged_path = INTERIM_DIR / "credit_risk_merged_interim.csv"
merged_df.to_csv(merged_path, index=False)
merged_path

WindowsPath('D:/Banking and Finance/Projects/canadian-retail-credit-risk-xai/data/interim/credit_risk_merged_interim.csv')

## Initial observations to carry forward

1. Each source sheet has the same row count.
2. `user_id` is not unique; preserve `record_sequence`.
3. The safe merged dataset should match the original source row count.
4. Missingness is material in employment, marital, social, verification, and loan amount fields.
5. Repayment and delinquency variables require leakage review before modelling.
6. Gender, marital status, dependents, home ownership, postal/pincode, and social profile require fairness/proxy review.